# Cyclopropene in VibeMol

Compute the two lowest singlet states with equal weights using
SA2-CASSCF(4,3)/cc-pVDZ. AVAS selects the three active orbitals from the
plane of the carbon atoms. This notebook needs JupyterLab and
vibemol-widget 0.2.0 or later. Use **Run → Run All Cells** to compute
the results and display the four panels.

In [ ]:
import os

from IPython.display import display

from forte2 import AVAS, CISolver, MCOptimizer, RHF, State, System
from forte2.vibemol import (
    show_dipole,
    show_forces,
    show_orbitals,
    show_transition_dipole,
    transition_dipole,
)
from vibemol import Viewer

In [ ]:
CYCLOPROPENE_XYZ = """
H   0.912650   0.000000   1.457504
H  -0.912650   0.000000   1.457504
H   0.000000  -1.585659  -1.038624
H   0.000000   1.585659  -1.038624
C   0.000000   0.000000   0.859492
C   0.000000  -0.651229  -0.499559
C   0.000000   0.651229  -0.499559
"""
system = System(
    xyz=CYCLOPROPENE_XYZ,
    basis_set="cc-pvdz",
    auxiliary_basis_set="def2-universal-jkfit",
)
rhf = RHF(charge=0, e_tol=1e-10, d_tol=1e-8)(system)
avas = AVAS(
    subspace=["C(2p)"],
    subspace_pi_planes=[["C1-3"]],
    selection_method="total",
    num_active=3,
)(rhf)
ci = CISolver(
    State(nel=rhf.nel, multiplicity=1, ms=0.0), nroots=2, weights=[0.5, 0.5]
)
mc = MCOptimizer(
    ci,
    e_tol=1e-10,
    g_tol=1e-7,
    final_orbitals="original",
    do_transition_dipole=True,
)(avas).run()

In [ ]:
viewer = Viewer(url=os.environ.get("VIBEMOL_VIEWER_URL") or "https://vibemol.org/")
display(viewer)

## 1. Second active orbital

Load the Molden geometry and orbitals, then display the second active
orbital by its one-based Molden index.

In [ ]:
second_active = int(mc.mo_space.active_indices[1]) + 1
orbital = show_orbitals(
    viewer, mc, indices=[second_active], title="Cyclopropene SA2-CASSCF"
)
orbital_view = viewer.set_view(axis="+x")
_ = viewer.snapshot(width=640, height=460).display()

## 2. Forces on S1 and S0

Forces are the negative nuclear gradient, in Hartree per bohr.
Scale the largest S1 force arrow to 1 Å, then reuse that scale on
S0 so both states can be compared directly. Magenta marks S1 and
amber marks S0.

In [ ]:
hidden_orbitals = viewer.hide_orbitals()
force_view = viewer.set_view(axis="+x")
s1_forces = show_forces(viewer, mc, root=1, name="S1 forces")
s0_forces = show_forces(
    viewer, mc, root=0, scale=s1_forces.scale, name="S0 forces", color="#C57B20"
)
_ = viewer.snapshot(width=640, height=460).display()

## 3. Total state dipoles

Each state dipole includes the nuclear and electronic contributions,
in atomic units of electron charge times bohr. The S0 (violet) and
S1 (blue) arrows point in opposite directions and use the same scale,
with the S1 arrow drawn 1.5 Å long.

In [ ]:
cleared_forces = viewer.clear_vectors()
s1_dipole = show_dipole(viewer, mc, root=1, name="S1 dipole")
s0_dipole = show_dipole(
    viewer, mc, root=0, scale=s1_dipole.scale, name="S0 dipole", color="#8850A4"
)
_ = viewer.snapshot(width=640, height=460).display()

## 4. S0 → S1 transition dipole

The transition dipole is in atomic units and its overall sign is
arbitrary. The two-headed arrow shows its axis at a fixed length of
2.4 Å. The oscillator strength is dimensionless.

In [ ]:
cleared_dipoles = viewer.clear_vectors()
transition = show_transition_dipole(viewer, mc, 0, 1)
_, oscillator_strength = transition_dipole(mc, 0, 1)
print(f"S0 → S1 oscillator strength: {oscillator_strength:.6f}")
_ = viewer.snapshot(width=640, height=460).display()

## Snapshots from Run All

Captured in JupyterLab against vibemol.org using Forte2 `bdf950adee3c94d1b6b6957f5f907505f0f28708`, vibemol-widget 0.2.0, and VibeMol app 0.10.18.

### Second active orbital

![Second active cyclopropene orbital, MO 11](../img/vibemol_orbital.png)

### S1 and S0 forces

![Magenta S1 and amber S0 forces on the same scale](../img/vibemol_forces.png)

### State dipoles

![Opposite S0 and S1 dipole arrows on a common scale](../img/vibemol_dipoles.png)

### Transition dipole

![Two-headed S0 to S1 transition dipole](../img/vibemol_transition.png)